lista -> tf e idf do corpus

In [1]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

corpus = [
    "the sky is blue",
    "the sun is bright",
    "the sun in the sky",
]

def tokenizer(doc):
    return [w for w in word_tokenize(doc) if w not in stopwords.words('english')]

corpus_tokens = [tokenizer(doc) for doc in corpus]

print(corpus_tokens)


[['sky', 'blue'], ['sun', 'bright'], ['sun', 'sky']]


In [2]:
for d in corpus_tokens:
    for w in d:
        print(w)

sky
blue
sun
bright
sun
sky


In [3]:
from collections import Counter

def tf(t, d):
    N = len(d)
    count = 0
    for word in d:
        if word == t:
            count += 1
    return count / N

def doc_tf(doc):
    N = len(doc)
    counter = Counter(doc)
    for c in counter:
        counter[c] = counter[c] / N
    return counter
        
print(doc_tf(corpus_tokens[0]))

Counter({'sky': 0.5, 'blue': 0.5})


In [4]:
import math 

def idf(corpus_tokens):
    N = len(corpus_tokens) 
    res = {}
    for d in corpus_tokens:
        for t in set(d): 
            if t not in res:
                df = sum(1 for doc in corpus_tokens if t in doc)
                res[t] = math.log(N / df, 10) 
    return res

print(idf(corpus_tokens))

{'sky': 0.17609125905568124, 'blue': 0.47712125471966244, 'sun': 0.17609125905568124, 'bright': 0.47712125471966244}


In [5]:
# vamos agora criar a matriz TF-IDF

def tf_idf(corpus_tokens):
    idf_values = idf(corpus_tokens)
    matrix = []
    for doc in corpus_tokens:
        dict = {}
        tf_values = doc_tf(doc)
        for t in tf_values:
            dict[t] = tf_values[t] * idf_values[t]
        matrix.append(dict)
    return matrix

print(tf_idf(corpus_tokens))

[{'sky': 0.08804562952784062, 'blue': 0.23856062735983122}, {'sun': 0.08804562952784062, 'bright': 0.23856062735983122}, {'sun': 0.08804562952784062, 'sky': 0.08804562952784062}]


In [6]:
def vectorize(tf_idf_dict):
    vocab = set(token for d in corpus_tokens for token in d)
    res = []
    for doc in tf_idf_dict:
        res.append([doc.get(token, 0) for token in vocab])
    return res
    
    
print(vectorize(tf_idf(corpus_tokens)))
    

[[0, 0.08804562952784062, 0.23856062735983122, 0], [0.08804562952784062, 0, 0, 0.23856062735983122], [0.08804562952784062, 0.08804562952784062, 0, 0]]


In [7]:
query = "The bright sun"

query_tokens = tokenizer(query)

# fazer information retrieval usando a matriz TF-IDF

def cosine_similarity(vec_a, vec_b):
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a ** 2 for a in vec_a))
    norm_b = math.sqrt(sum(b ** 2 for b in vec_b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

def vectorize_query(query_tokens, corpus_tokens, idf_values):
    vocab = list(set(token for d in corpus_tokens for token in d))
    tf_values = doc_tf(query_tokens)
    vec = []
    for token in vocab:
        tf_val = tf_values.get(token, 0)
        idf_val = idf_values.get(token, 0)  # palavras fora do corpus têm idf=0
        vec.append(tf_val * idf_val)
    return vec, vocab

idf_values = idf(corpus_tokens)
tfidf_matrix = tf_idf(corpus_tokens)

query_vec, vocab = vectorize_query(query_tokens, corpus_tokens, idf_values)
doc_vecs = vectorize(tfidf_matrix)

In [8]:
scores = []
for i, doc_vec in enumerate(doc_vecs):
    score = cosine_similarity(query_vec, doc_vec)
    scores.append((i, score, corpus[i]))

# Ordenar por relevância (decrescente)
scores.sort(key=lambda x: x[1], reverse=True)

In [9]:
for rank, (doc_idx, score, doc_text) in enumerate(scores, 1):
    print(f"  {rank}. [score={score:.4f}] Doc {doc_idx}: \"{doc_text}\"")

  1. [score=1.0000] Doc 1: "the sun is bright"
  2. [score=0.2448] Doc 2: "the sun in the sky"
  3. [score=0.0000] Doc 0: "the sky is blue"
